<a href="https://colab.research.google.com/github/Anorwed/vkr/blob/main/multiqwen2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Для запуска нажмите кнопку слева от "Показать код"
import os
import re
import time
import json
import subprocess
import sys
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path

# === ФИКСИРОВАННЫЕ ВЕРСИИ ===
PICOCLAW_VERSION = "0.2.8"
PICOCLAW_DOWNLOAD_URL = f"https://github.com/sipeed/picoclaw/releases/download/v{PICOCLAW_VERSION}/picoclaw_Linux_x86_64.tar.gz"

MODEL_CHOICES = {
    "Qwen 3.5 (9B) — max качество, медленно на T4": "qwen3.5:9b",
    "Qwen 3.5 (4B) — баланс скорости и качества": "qwen3.5:4b",
    "Qwen 2.5 (7B) — быстрее 9B, хорошее качество": "qwen2.5:7b",
}

def run_installation(vk_token, community_url, model_name):
    vk_token = vk_token.strip()
    community_url = community_url.strip()

    if not vk_token:
        print("❌ Ошибка: Токен сообщества VK не может быть пустым!")
        return
    if not community_url:
        print("❌ Ошибка: Ссылка на сообщество VK не может быть пустой!")
        return

    group_id = None
    m = re.search(r'(?:vk\.com|vk\.ru)/(?:club|public)(\d+)', community_url)
    if not m:
        m = re.search(r'^(?:club|public)(\d+)$', community_url)
    if m:
        group_id = int(m.group(1))
    else:
        print("❌ Ошибка: Не удалось извлечь ID сообщества.")
        print("   Форматы: https://vk.com/club123456789 или public123456789")
        return

    print(f"=== PicoClaw + Ollama + VK ===")
    print(f"🆔 VK Group ID: {group_id}")
    print(f"🤖 Модель: {model_name}")
    print(f"📦 PicoClaw: v{PICOCLAW_VERSION} (фиксированная версия)")
    print(f"🔧 Подготовка системы...")
    subprocess.run('apt-get update -qq && apt-get install -y -qq zstd pciutils net-tools',
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print(f"🦙 Установка Ollama...")
    subprocess.run('curl -fsSL https://ollama.com/install.sh | sh',
                   shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    print(f"🚀 Запуск Ollama...")
    os.environ['OLLAMA_HOST'] = '0.0.0.0:11434'
    subprocess.Popen(['nohup', 'ollama', 'serve'], stdout=open('ollama.log', 'w'), stderr=subprocess.STDOUT)
    time.sleep(12)

    print(f"🔍 Проверка Ollama...")
    for i in range(6):
        try:
            import urllib.request
            with urllib.request.urlopen('http://localhost:11434/api/tags', timeout=3) as r:
                if r.status == 200:
                    print("   ✅ Ollama отвечает")
                    break
        except Exception:
            time.sleep(2)
    else:
        print("   ⚠️ Ollama не отвечает, продолжаем...")

    print(f"📥 Загрузка {model_name}...")
    subprocess.run(['ollama', 'pull', model_name])

    print(f"🔍 Проверка VRAM...")
    try:
        ps_out = subprocess.check_output(['ollama', 'ps'], text=True, stderr=subprocess.STDOUT)
        print(ps_out)
        if '100% CPU' in ps_out:
            print("   ⚠️ Модель на CPU — будет очень медленно!")
    except Exception:
        print("   ℹ️ Не удалось проверить статус GPU")

    print(f"🧪 Тест генерации...")
    try:
        import urllib.request
        req = urllib.request.Request(
            'http://localhost:11434/api/generate',
            data=json.dumps({
                "model": model_name,
                "prompt": "Say OK",
                "stream": False,
                "options": {"num_predict": 5}
            }).encode(),
            headers={'Content-Type': 'application/json'},
            method='POST'
        )
        with urllib.request.urlopen(req, timeout=60) as r:
            resp = json.loads(r.read().decode())
            if resp.get('response', '').strip():
                print("   ✅ Модель работает")
            else:
                print("   ⚠️ Пустой ответ")
    except Exception as e:
        print(f"   ⚠️ Тест не пройден: {e}")

    # === ЗАГРУЗКА ФИКСИРОВАННОЙ ВЕРСИИ PICOCLAW ===
    print(f"📥 Загрузка PicoClaw v{PICOCLAW_VERSION}...")
    print(f"   URL: {PICOCLAW_DOWNLOAD_URL}")

    result = subprocess.run(
        ['wget', '-q', PICOCLAW_DOWNLOAD_URL],
        capture_output=True, text=True
    )
    if result.returncode != 0:
        print(f"   ❌ Ошибка загрузки: {result.stderr}")
        print(f"   Проверьте, существует ли релиз v{PICOCLAW_VERSION}")
        return

    subprocess.run('tar -xzf picoclaw_Linux_x86_64.tar.gz', shell=True, check=True)
    subprocess.run('chmod +x picoclaw', shell=True)

    print(f"🔍 Проверка PicoClaw...")
    try:
        ver = subprocess.check_output(['./picoclaw', 'version'], text=True, stderr=subprocess.STDOUT).strip()
        print(f"   ✅ {ver}")
    except Exception:
        ver = subprocess.check_output(['./picoclaw', '--version'], text=True, stderr=subprocess.STDOUT).strip()
        print(f"   ✅ {ver}")

    os.makedirs("/content/workspace", exist_ok=True)
    os.makedirs("/root/.picoclaw", exist_ok=True)

    config_path = Path.home() / ".picoclaw" / "config.json"

    config_data = {
        "version": 3,
        "agents": {
            "defaults": {
                "workspace": "/content/workspace",
                "model_name": "qwen-local",
                "max_tokens": 1024,
                "temperature": 0.7,
                "max_tool_iterations": 10,
                "restrict_to_workspace": True
            }
        },
        "model_list": [
            {
                "model_name": "qwen-local",
                "model": f"ollama/{model_name}"
            }
        ],
        "channel_list": {
            "vk": {
                "enabled": True,
                "type": "vk",
                "allow_from": [],
                "typing": {"enabled": False},
                "settings": {
                    "token": vk_token,
                    "group_id": group_id
                }
            }
        },
        "tools": {
            "cron": {"enabled": False},
            "spawn": {"enabled": False},
            "spawn_status": {"enabled": False},
            "subagent": {"enabled": False},
            "find_skills": {"enabled": False},
            "install_skill": {"enabled": False},
            "i2c": {"enabled": False},
            "spi": {"enabled": False},
            "media_cleanup": {"enabled": False},
            "mcp": {"enabled": False},
            "exec": {"enabled": True},
            "message": {"enabled": True},
            "read_file": {"enabled": True},
            "write_file": {"enabled": True},
            "edit_file": {"enabled": True},
            "append_file": {"enabled": True},
            "list_dir": {"enabled": True},
            "send_file": {"enabled": True},
            "web_fetch": {"enabled": True}
        },
        "gateway": {
            "host": "0.0.0.0",
            "port": 18790,
            "log_level": "debug"
        }
    }

    with open(config_path, "w") as f:
        json.dump(config_data, f, indent=4)

    diag = json.loads(json.dumps(config_data))
    diag["channel_list"]["vk"]["settings"]["token"] = vk_token[:10] + "..." if len(vk_token) > 10 else "***"
    print(f"\n📄 Конфиг:")
    print(json.dumps(diag, indent=2))

    print(f"\n✅ Конфиг записан")
    print(f"📋 Модель: ollama/{model_name} | max_tokens: 1024")
    print(f"📦 PicoClaw: v{PICOCLAW_VERSION} (фиксированная)")
    print(f"🛠️  Системный промпт сокращён (отключены: cron, spawn, subagent, skills, i2c, spi, mcp)")
    print(f"💬 Канал: VK (group_id={group_id})")
    print(f"\n🚀 Запуск PicoClaw Gateway...")
    print(f"   (для остановки: Ctrl+M I)\n")
    print("-" * 50)

    env = os.environ.copy()
    env['PICOCLAW_LOG_LEVEL'] = 'debug'

    process = subprocess.Popen(
        ['./picoclaw', 'gateway'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
        bufsize=1,
        universal_newlines=True
    )

    try:
        for line in iter(process.stdout.readline, ''):
            if not line:
                break
            print(line, end='')
            sys.stdout.flush()
    except KeyboardInterrupt:
        print("\n🛑 Остановка...")
    finally:
        process.stdout.close()
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
        print(f"\n⚠️ PicoClaw завершился с кодом: {process.returncode}")

# --- Интерфейс ---
model_dropdown = widgets.Dropdown(
    options=MODEL_CHOICES,
    value="qwen3.5:4b",
    description='Модель:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

token_input = widgets.Password(
    value='',
    placeholder='vk1.a.xxx... (токен сообщества)',
    description='VK Token:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

url_input = widgets.Text(
    value='',
    placeholder='https://vk.com/club123456789',
    description='Ссылка:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='500px')
)

start_button = widgets.Button(
    description='🚀 Запустить',
    button_style='success',
    icon='play',
    layout=widgets.Layout(width='180px')
)

output = widgets.Output()

def on_button_clicked(b):
    with output:
        clear_output()
        run_installation(token_input.value, url_input.value, model_dropdown.value)

start_button.on_click(on_button_clicked)

display(widgets.VBox([
    widgets.HTML("<h3>🦞 PicoClaw + Ollama + VK Сообщество</h3>"),
    widgets.HTML("<p>Выберите модель, введите токен и ссылку на сообщество</p>"),
    model_dropdown,
    token_input,
    url_input,
    start_button,
    output
]))
